# NyaayKhel — 00: Kabaddi Clip Downloader

**Purpose:** Download 150–300 short (2–10 sec) kabaddi clips from YouTube for training data.

**What this notebook produces:**
- `data/raw/<angle_bucket>/<clip_id>.mp4` — trimmed short clips, ready for labeling
- `data/raw/clip_index.csv` — metadata: clip ID, source URL, angle bucket, duration

**Ground rules (read before running):**
- All footage is from publicly available YouTube videos. Document source URLs in `clip_index.csv`.
- Target angle distribution: ~50% side-view (~90°), ~30% quarter-view (~70°), ~20% angled (~60°)
- Keep clips short (2–10 sec). Labeling is faster with short clips.
- Quality > quantity. 150 well-labeled clips > 300 noisy ones.

**Exit gate:** 150+ clips downloaded and organised into angle buckets.

## Cell 1: Install Dependencies

In [ ]:
# Install yt-dlp (YouTube downloader) and ffmpeg-python
!pip install -q yt-dlp ffmpeg-python

# Verify ffmpeg is available (Colab usually has it)
import subprocess
result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
print('ffmpeg:', result.stdout.split('\n')[0] if result.returncode == 0 else 'NOT FOUND — install with: apt-get install ffmpeg')

## Cell 2: Mount Google Drive (optional — for persistent storage across sessions)

In [ ]:
# Mount Google Drive so clips persist across Colab sessions.
# Skip this cell if you want to store clips locally in Colab's /content/ (lost on disconnect).

USE_DRIVE = True  # Set False to store in /content/ only

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/NyaayKhel'
else:
    BASE_DIR = '/content/NyaayKhel'

import os
RAW_DIR = os.path.join(BASE_DIR, 'data', 'raw')
for bucket in ['side_90', 'quarter_70', 'angled_60', 'misc']:
    os.makedirs(os.path.join(RAW_DIR, bucket), exist_ok=True)

print(f'Storage base: {BASE_DIR}')
print(f'Raw clip dirs created under: {RAW_DIR}')

## Cell 3: Define Search Terms & Download Config

In [ ]:
import csv, os, time, hashlib
from pathlib import Path

# ---------------------------------------------------------------------------
# TARGET: 150–300 clips across 4 classes (raid_start, touch, escape_return, neutral)
# Each "video" below will be downloaded, then sliced into short clips.
# You'll label individual clips in CVAT (Phase B).
# ---------------------------------------------------------------------------

# YouTube search queries — yt-dlp supports 'ytsearch<N>:query' syntax
# Each query downloads N videos matching the search.
# We then slice each video into short clips.
SEARCH_QUERIES = [
    # Side-view (~90°) — most important for training
    ('ytsearch5:kabaddi match full tournament side view', 'side_90'),
    ('ytsearch5:kabaddi raid touch district tournament', 'side_90'),
    ('ytsearch5:kabaddi village match full video India', 'side_90'),
    ('ytsearch3:kabaddi grassroots match recording', 'side_90'),
    # Quarter-view (~60-70°)
    ('ytsearch4:kabaddi tournament state level match', 'quarter_70'),
    ('ytsearch3:pro kabaddi training footage', 'quarter_70'),
    # Misc angles — for robustness
    ('ytsearch3:kabaddi national championship full match', 'misc'),
    ('ytsearch3:kho kho kabaddi wrestling grassroots sport India', 'misc'),
]

# Max video duration to download (seconds) — skip full 90-min matches
# We want highlight/tournament clips, not full broadcast matches
MAX_VIDEO_DURATION_SEC = 600  # 10 minutes

# Clip slicing config (applied AFTER download)
CLIP_DURATION_SEC = 5    # Each output clip length
CLIP_STRIDE_SEC = 3      # Stride between clip starts (overlap = 2 sec)

print(f'Configured {len(SEARCH_QUERIES)} search queries')
print(f'Max video duration: {MAX_VIDEO_DURATION_SEC}s | Clip: {CLIP_DURATION_SEC}s | Stride: {CLIP_STRIDE_SEC}s')

## Cell 4: Download Videos

In [ ]:
import yt_dlp
import json

DOWNLOAD_DIR = os.path.join(BASE_DIR, 'data', 'raw', 'full_videos')
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# Index file to track sources
INDEX_PATH = os.path.join(RAW_DIR, 'download_index.csv')
downloaded_videos = []  # will hold metadata dicts

ydl_opts = {
    'format': 'bestvideo[height<=720][ext=mp4]+bestaudio[ext=m4a]/best[height<=720][ext=mp4]/best[height<=720]',
    'outtmpl': os.path.join(DOWNLOAD_DIR, '%(id)s.%(ext)s'),
    'noplaylist': True,
    'quiet': False,
    'no_warnings': False,
    'match_filter': yt_dlp.utils.match_filter_func(f'duration < {MAX_VIDEO_DURATION_SEC}'),
    'writeinfojson': True,  # saves <id>.info.json alongside each video
    'ignoreerrors': True,   # skip unavailable videos
}

print(f'Downloading to: {DOWNLOAD_DIR}')
print('This will take several minutes depending on connection speed.\n')

for query, bucket in SEARCH_QUERIES:
    print(f'--- Query: "{query}" → bucket: {bucket} ---')
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(query, download=True)
            if info and 'entries' in info:
                for entry in info['entries']:
                    if entry is None:
                        continue
                    video_id = entry.get('id', 'unknown')
                    duration = entry.get('duration', 0)
                    title = entry.get('title', '')
                    webpage_url = entry.get('webpage_url', '')
                    downloaded_videos.append({
                        'video_id': video_id,
                        'title': title,
                        'url': webpage_url,
                        'duration_sec': duration,
                        'angle_bucket': bucket,
                        'file': os.path.join(DOWNLOAD_DIR, f'{video_id}.mp4'),
                    })
                    print(f'  ✓ {video_id} | {duration}s | {title[:60]}')
    except Exception as e:
        print(f'  ✗ Failed query "{query}": {e}')
    time.sleep(1)  # polite delay between queries

# Write download index
if downloaded_videos:
    with open(INDEX_PATH, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=downloaded_videos[0].keys())
        writer.writeheader()
        writer.writerows(downloaded_videos)
    print(f'\nDownload index written to {INDEX_PATH}')

print(f'\nTotal videos downloaded: {len(downloaded_videos)}')

## Cell 5: Slice Videos into Short Clips

In [ ]:
import subprocess
import uuid

# ---------------------------------------------------------------------------
# Slice each downloaded video into CLIP_DURATION_SEC clips.
# Output naming: <video_id>_t<start_sec>.mp4
# Output goes into the angle-bucket folder for that video.
# ---------------------------------------------------------------------------

CLIP_INDEX_PATH = os.path.join(RAW_DIR, 'clip_index.csv')
clip_records = []
total_clips = 0
skipped = 0

for vid in downloaded_videos:
    src = vid['file']
    if not os.path.exists(src):
        print(f'Skip (file missing): {src}')
        skipped += 1
        continue

    bucket_dir = os.path.join(RAW_DIR, vid['angle_bucket'])
    duration = vid['duration_sec'] or 0
    if duration < 3:
        print(f'Skip (too short): {vid["video_id"]} ({duration}s)')
        skipped += 1
        continue

    t = 0
    while t + CLIP_DURATION_SEC <= duration:
        clip_id = f"{vid['video_id']}_t{int(t):05d}"
        out_path = os.path.join(bucket_dir, f'{clip_id}.mp4')

        if not os.path.exists(out_path):
            cmd = [
                'ffmpeg', '-y',
                '-ss', str(t),
                '-i', src,
                '-t', str(CLIP_DURATION_SEC),
                '-c:v', 'libx264',
                '-preset', 'fast',
                '-crf', '23',
                '-an',           # drop audio — not needed for pose extraction
                '-vf', 'scale=640:-2',  # downsample to 640px wide
                out_path
            ]
            result = subprocess.run(cmd, capture_output=True)
            if result.returncode != 0:
                print(f'  ffmpeg error on {clip_id}: {result.stderr[-200:]}')

        clip_records.append({
            'clip_id': clip_id,
            'source_video_id': vid['video_id'],
            'source_url': vid['url'],
            'start_sec': t,
            'duration_sec': CLIP_DURATION_SEC,
            'angle_bucket': vid['angle_bucket'],
            'file': out_path,
            'label': '',  # filled in during Phase B labeling
        })
        total_clips += 1
        t += CLIP_STRIDE_SEC

# Write clip index
with open(CLIP_INDEX_PATH, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=clip_records[0].keys() if clip_records else ['clip_id'])
    writer.writeheader()
    writer.writerows(clip_records)

print(f'Total clips generated: {total_clips}')
print(f'Skipped videos: {skipped}')
print(f'Clip index written to: {CLIP_INDEX_PATH}')

# Angle bucket summary
from collections import Counter
bucket_counts = Counter(r['angle_bucket'] for r in clip_records)
print('\nClips per angle bucket:')
for bucket, count in bucket_counts.items():
    print(f'  {bucket}: {count} clips')

## Cell 6: Quick Visual Review — Spot-Check Clips

In [ ]:
# Display a random sample of clips as thumbnails to sanity-check quality.
# Run this in Colab — it renders inline images.

import random
import subprocess
from IPython.display import Image, display

SAMPLE_N = min(12, len(clip_records))
sample = random.sample(clip_records, SAMPLE_N)

THUMB_DIR = '/tmp/thumbs'
os.makedirs(THUMB_DIR, exist_ok=True)

print(f'Showing {SAMPLE_N} random clip thumbnails (frame at 1 sec):\n')
for r in sample:
    if not os.path.exists(r['file']):
        continue
    thumb_path = os.path.join(THUMB_DIR, r['clip_id'] + '.jpg')
    subprocess.run([
        'ffmpeg', '-y', '-ss', '1', '-i', r['file'],
        '-frames:v', '1', '-q:v', '3', thumb_path
    ], capture_output=True)
    if os.path.exists(thumb_path):
        print(f"  {r['clip_id']} | bucket: {r['angle_bucket']}")
        display(Image(thumb_path, width=320))

print('\n--- Review complete ---')
print('MANUAL STEP: Delete clips that are blurry, not kabaddi, or wrong angle before labeling.')

## Cell 7: Prepare Clip List for CVAT / Label Studio

After reviewing clips above, use this cell to generate the import file for your labeling tool.

In [ ]:
# Reload clip index (in case you cleaned up bad clips manually)
import csv

with open(CLIP_INDEX_PATH, 'r', encoding='utf-8') as f:
    clips = list(csv.DictReader(f))

# Filter to only clips that actually exist
valid_clips = [c for c in clips if os.path.exists(c['file'])]
print(f'Valid clips ready for labeling: {len(valid_clips)}')

# Export paths list for CVAT upload
paths_file = os.path.join(RAW_DIR, 'cvat_import_paths.txt')
with open(paths_file, 'w') as f:
    for c in valid_clips:
        f.write(c['file'] + '\n')

print(f'CVAT import paths written to: {paths_file}')
print()
print('LABEL CLASSES (use exactly these strings in CVAT/Label Studio):')
for cls in ['raid_start', 'touch', 'escape_return', 'neutral']:
    print(f'  • {cls}')
print()
print('Target distribution (rough guide):')
print('  raid_start:    ~30%')
print('  touch:         ~20%')
print('  escape_return: ~20%')
print('  neutral:       ~30%')
print()
print('→ Next step: Upload clips to CVAT/Label Studio and label each clip.')
print('  After labeling, export annotations as CSV or JSON and save to data/processed/')
print('  Then run notebook 02_dataset_builder.ipynb.')

## Exit Gate Checklist

Before moving to Phase B (labeling + model training):

- [ ] `data/raw/download_index.csv` exists with ≥ 15 source videos
- [ ] `data/raw/clip_index.csv` exists with ≥ 150 clip entries
- [ ] At least 100 clips actually exist on disk and pass visual review
- [ ] Clips are distributed across `side_90/`, `quarter_70/`, `angled_60/` buckets
- [ ] `data/raw/cvat_import_paths.txt` exported for labeling tool
- [ ] Notebook committed to GitHub with outputs cleared